# Treated MonoCulture Hill Modelling

Model treated cell lines using Hill equations, initialized from untreated monoculture parameters, and quantify IC shifts.

## 1) Import Packages and Configure Notebook

We'll use numpy, pandas, scipy.optimize, matplotlib, and seaborn. We'll also set a seed for reproducibility.

## 2) Load Base Utilities from Full Base Code

Attempt to import shared helpers from the base notebook; if not available as python modules, this cell is a placeholder.

## 3) Define Paths and Load Data

We'll load untreated parameter estimates (if available) and treated Treated MonoCulture datasets.

## 4) Inspect and Plot Raw Dose-Response Data

We’ll inspect counts and plot time trends for Mean Cells and SEM by Day.

## 5) Hill Equation Definitions and Helpers

We'll use the 4-parameter Hill model: E(c) = E_min + (E_max - E_min) / (1 + (c/IC50)^n).

## 6) Initialize Treated Model Parameters from Untreated

Join treated with untreated parameter estimates or use placeholders if not available.

## 7) Fit Treated Dose-Response Models

Fit [Emin, Emax, n, delta] with IC50_treated = IC50_untreated * exp(delta).

## 8) Compute IC Metrics and IC Shift vs Untreated

Compute IC10/50/90 for treated and fold-changes.

## 9) Visualize Fits and IC Shifts

Overlay fitted curves on observed data and show fold-change bars.

## 10) Bootstrap Confidence Intervals (optional)

Placeholder for bootstrap resampling by day/replicate if concentration-level data are added.

## 11) Save Fitted Parameters and Plots

Write results to CSV under `Modelling Data Notebooks/Treated MonoCulture/params/`.

## 1) Import Packages and Configure (Julia)

This notebook follows the structure of `Full Base Code.ipynb` but focuses on treated monocultures and Hill modeling.

In [ ]:
# Activate and import packages
using Pkg
const INSTALL_PKGS = false  # set to true once if packages are missing
if INSTALL_PKGS
    Pkg.add(["CSV", "DataFrames", "StatsBase", "LsqFit", "Plots", "StatsPlots"]) 
end

using CSV
using DataFrames
using StatsBase
using LsqFit
using Plots
using StatsPlots  # for groupedbar

# Plot theme
default(fmt=:png, legend=:topright, size=(900, 550))

## 2) Load Base Utilities from Full Base Code (Julia)

Bring in any shared functions by including relevant cells or files referenced in `Full Base Code.ipynb`.

In [ ]:
# If utilities were factored into a .jl file, include it here.
# For example: include(joinpath(@__DIR__, "..", "..", "FullBaseUtils.jl"))
# Otherwise, we'll reimplement minimal helpers used below.

# Basic helper to safely read a CSV into a DataFrame
safe_read_csv(path::AbstractString) = begin
    if isfile(path)
        CSV.read(path, DataFrame)
    else
        @warn "Missing file" path
        DataFrame()
    end
end

## 3) Define Paths and Load Data

Read processed Treated MonoCulture datasets and (optionally) untreated parameter estimates for initialization.

In [ ]:
# Paths
ROOT = raw"c:/Users/MainFrameTower/Desktop/CancerGrowthDynamics"
UNTREATED_PARAM_PATH = joinpath(ROOT, "Modelling Data Notebooks", "Untreated MonoCulture", "untreated_params.csv")
TRE_ROOT = joinpath(ROOT, "Processed_Datasets", "Treated MonoCulture", "30k")

function load_ic_level(ic::AbstractString)
    folder = joinpath(TRE_ROOT, ic, "Averages")
    files = Dict(
        "A2780Naive_day" => joinpath(folder, "A2780Naive_day_averages.csv"),
        "A2780Naive_sample" => joinpath(folder, "A2780Naive_sample_averages.csv"),
        "A2780cis_day" => joinpath(folder, "A2780cis_day_averages.csv"),
        "A2780cis_sample" => joinpath(folder, "A2780cis_sample_averages.csv")
    )
    Dict(k => safe_read_csv(p) for (k,p) in files)
end

ic25 = load_ic_level("IC25")
ic50 = isdir(joinpath(TRE_ROOT, "IC50")) ? load_ic_level("IC50") : Dict{String,DataFrame}()
ic75 = isdir(joinpath(TRE_ROOT, "IC75")) ? load_ic_level("IC75") : Dict{String,DataFrame}()

untreated_params = isfile(UNTREATED_PARAM_PATH) ? safe_read_csv(UNTREATED_PARAM_PATH) : DataFrame()

# Quick peek at any non-empty DataFrame
for (k, df) in ic25
    if nrow(df) > 0
        @info "Preview" key=k
        display(first(df, 5))
        break
    end
end

## 4) Inspect and Plot Raw Time Series

Plot Mean Cells over Day with error ribbons.

In [ ]:
function plot_time_series(df::DataFrame, title::AbstractString)
    if nrow(df) == 0
        @warn "No data to plot" title
        return
    end
    required = ["Day", "Mean Cells"]
    all(in.(required, Ref(names(df)))) || begin
        @warn "Missing required columns" names(df)
        return
    end
    x = df."Day"
    y = df."Mean Cells"
    p = plot(x, y, label="Mean", lw=2, marker=:circle, xlabel="Day", ylabel="Mean Cells", title=title)
    if "SEM Cells" in names(df)
        sem = df."SEM Cells"
        ribbon = sem
        plot!(p, x, y, ribbon=ribbon, fillalpha=0.2, label="± SEM")
    end
    display(p)
end

plot_time_series(get(ic25, "A2780Naive_sample", DataFrame()), "IC25 A2780Naive (Sample)")
plot_time_series(get(ic25, "A2780cis_sample", DataFrame()), "IC25 A2780cis (Sample)")

## 5) Hill Equation Definitions (Julia)

We use the 4-parameter Hill model: E(c) = E_min + (E_max - E_min) / (1 + (c/IC50)^n).

In [ ]:
hill4p(c, Emin, Emax, IC50, n) = Emin .+ (Emax .- Emin) ./ (1 .+ (c ./ IC50) .^ n)

# Treated parameterization: IC50_treated = IC50_untreated * exp(delta)
hill4p_shift(c, Emin, Emax, n, delta, IC50_untreated) = hill4p(c, Emin, Emax, IC50_untreated * exp(delta), n)

function ic_at_effect(Etarget, Emin, Emax, IC50, n)
    # Solve E(c) = Etarget for c (monotonic portions)
    Etarget = clamp(Etarget, min(Emin, Emax) + 1e-12, max(Emin, Emax) - 1e-12)
    ratio = (Emax - Emin) / (Etarget - Emin) - 1.0
    return IC50 * ratio^(1/n)
end

## 6) Initialize From Untreated Parameters

Use untreated parameters per cell line if available; otherwise use sensible defaults.

In [ ]:
function get_untreated_guess(cell_line::AbstractString)
    if nrow(untreated_params) > 0 && all(in.( ["cell_line","Emin","Emax","IC50","n"], Ref(names(untreated_params))))
        sub = untreated_params[untreated_params."cell_line" .== cell_line, :]
        if nrow(sub) > 0
            r = sub[1, :]
            return (; Emin = Float64(r.Emin), Emax = Float64(r.Emax), IC50 = Float64(r.IC50), n = Float64(r.n))
        end
    end
    return (; Emin = 0.0, Emax = 2000.0, IC50 = 1.0, n = 1.0)
end

guess_naive = get_untreated_guess("A2780Naive")
guess_cis   = get_untreated_guess("A2780cis")

(@show guess_naive; @show guess_cis;)

## 7) Fit Treated Models (IC shift)

For each cell line and IC level, fit parameters [Emin, Emax, n, delta] where IC50_treated = IC50_untreated * exp(delta).

In [ ]:
# Note: The processed CSVs are time-series; dose information per time point is not included.
# We'll model per IC level using a placeholder constant concentration c = {0.25, 0.5, 0.75}.
# Replace with actual concentration vectors if available.

const IC_MAP = Dict("IC25"=>0.25, "IC50"=>0.50, "IC75"=>0.75)

function fit_ic_level(cell_line::AbstractString, ic_label::AbstractString, df::DataFrame, guess)
    if nrow(df) == 0
        return DataFrame()
    end
    y = Float64.(df."Mean Cells")
    c = fill(IC_MAP[ic_label], length(y))

    # params: p = [Emin, Emax, n, delta]
    model(p, c) = hill4p_shift(c, p[1], p[2], p[3], p[4], guess.IC50)
    p0 = [guess.Emin, guess.Emax, max(0.5, guess.n), 0.0]

    # Bounds via transformation (simple clamp after fit); LsqFit doesn't natively do bounds.
    fit = curve_fit(model, c, y, p0)
    p = coef(fit)
    Emin, Emax, n, delta = p
    IC50_treated = guess.IC50 * exp(delta)

    DataFrame(
        cell_line = [cell_line],
        ic_label  = [ic_label],
        Emin = [Emin], Emax = [Emax], n = [n], delta = [delta],
        IC50_untreated = [guess.IC50], IC50_treated = [IC50_treated],
        fold_change = [exp(delta)]
    )
end

fits = DataFrame()
if haskey(ic25, "A2780Naive_sample")
    fits = vcat(fits, fit_ic_level("A2780Naive", "IC25", ic25["A2780Naive_sample"], guess_naive))
end
if haskey(ic25, "A2780cis_sample")
    fits = vcat(fits, fit_ic_level("A2780cis", "IC25", ic25["A2780cis_sample"], guess_cis))
end

first(fits, 5) |> display

## 8) Compute IC Metrics and Shifts

In [ ]:
function compute_ic_metrics_row(Emin, Emax, n, IC50)
    ic10 = ic_at_effect(Emin + 0.9*(Emax-Emin), Emin, Emax, IC50, n)
    ic90 = ic_at_effect(Emin + 0.1*(Emax-Emin), Emin, Emax, IC50, n)
    (; ic10, ic90)
end

if nrow(fits) > 0
    ic10s = Float64[]
    ic90s = Float64[]
    for r in eachrow(fits)
        m = compute_ic_metrics_row(r.Emin, r.Emax, r.n, r.IC50_treated)
        push!(ic10s, m.ic10)
        push!(ic90s, m.ic90)
    end
    fits.IC10_treated = ic10s
    fits.IC90_treated = ic90s
end

display(fits)

## 9) Visualize Parameter Shifts

Plot log2 fold-change of IC50 vs untreated.

In [ ]:
if nrow(fits) > 0
    fits.log2_fold = log2.(Float64.(fits.fold_change))
    p = groupedbar(fits.cell_line, fits.log2_fold, group=fits.ic_label,
                   xlabel="Cell line", ylabel="log2 fold IC50 (treated/untreated)",
                   title="IC50 shift vs untreated", bar_position=:dodge)
    hline!(p, [0.0], lc=:black, lw=1, ls=:dash, label="no shift")
    display(p)
end

## 10) Save Parameters and Figures

In [ ]:
params_dir = joinpath(ROOT, "Modelling Data Notebooks", "Treated MonoCulture", "params")
if !isdir(params_dir)
    mkpath(params_dir)
end
out_path = joinpath(params_dir, "treated_fit_summary.csv")

if nrow(fits) > 0
    CSV.write(out_path, fits)
    @info "Saved fitted parameters" out_path
else
    @warn "No fitted results to save."
end